### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

from data.crisismmd import (
    load_crisismmd_annotations,
    CrisisVisionDataset,
    make_train_transforms,
    make_eval_transforms,
)
from models.vision_branch import build_resnet_classifier

### Load CrisisMMD dataset

In [ ]:
root = "../data/CrisisMMD_v2.0"  # folder containing annotations/ and data_image/
combined = load_crisismmd_annotations(root)

dataset = combined[["tweet_id", "image_id", "text_info", "image_info", "tweet_text", "image_path"]]
train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=42)

train_tf = make_train_transforms()
test_tf  = make_eval_transforms()

train_dataset = CrisisVisionDataset(train_df, root_dir=root, transform=train_tf)
test_dataset  = CrisisVisionDataset(test_df, root_dir=root, transform=test_tf)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

### Build model and optimizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_resnet_classifier(num_classes=2, pretrained=True).to(device)

# Freeze all params, then unfreeze layer4 + fc like in your original notebook
for p in model.parameters():
    p.requires_grad = False
for p in model.layer4.parameters():
    p.requires_grad = True
for p in model.fc.parameters():
    p.requires_grad = True

optimizer = optim.Adam(
    [
        {"params": model.layer4.parameters(), "lr": 1e-4},
        {"params": model.fc.parameters(), "lr": 1e-3},
    ]
)
criterion = nn.CrossEntropyLoss()

EPOCHS = 3

train_loss_history = []
train_acc_history = []

### Train the model

In [ ]:
print(f"Using device: {device}")
print("Starting ResNet training...")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, leave=True)
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        acc = 100.0 * correct / total
        loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}]")
        loop.set_postfix(loss=loss.item(), acc=acc)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    train_loss_history.append(epoch_loss)
    train_acc_history.append(epoch_acc)

print("Training complete.")


### Save model

In [ ]:
os.makedirs("../checkpoints", exist_ok=True)
torch.save(model.state_dict(), "../checkpoints/vision_brain.pth")
print("Saved vision model to ../checkpoints/vision_brain.pth")

### Plot training curves

In [ ]:
epochs = np.arange(1, EPOCHS + 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(epochs, train_loss_history, marker="o")
ax[0].set_title("Training Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")

ax[1].plot(epochs, train_acc_history, marker="o")
ax[1].set_title("Training Accuracy")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")

plt.tight_layout()
plt.show()

### Evaluation and confusion matrix

In [ ]:
model.eval()
all_preds = []
all_true  = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

acc = accuracy_score(all_true, all_preds)
f1  = f1_score(all_true, all_preds, average="binary")

print("\n--- VISION MODEL RESULTS ---")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Score: {f1:.4f}")

cm = confusion_matrix(all_true, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Safe (0)", "Disaster (1)"])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap="Reds", ax=ax, colorbar=False)
plt.title("ResNet Vision Model - Confusion Matrix (Test)", fontsize=14)
plt.grid(False)
plt.show()